# GIN — Recipe Tuning on Config 2 (HI-Small)

- **Question:** which training recipe (dropout, normalisation, depth, class weight, learning rate, FFN) should the GIN family use? Tuned **once** here on config 2 (mp base / readout base), then frozen for every GIN feature config.
- **Protocol (`EXPERIMENTS.md` §2b, fixed in advance):** one knob at a time in a pre-registered order — norm → layers → FFN → pos_weight → learning rate → dropout (architecture first, then optimizer, regularisation last) — starting from the v1 recipe; 2 seeds per trial, selection on validation only, cosine T_max 100 with early stopping, best epoch from the smoothed seed-mean validation curve; a trial replaces the incumbent only if it beats it by more than max(0.01, the larger seed std); the edge rule adds one trial one step beyond a winner at the edge of its grid.
- **Everything shared** lives in `gnn_core.py` (`run_tuning_trial`, `choose_winner`, `probe_depths`, `save_recipe`). This notebook only sets the operator, the stages and paths.
- **Never touches test:** only train and validation loaders are built, no `predictions.csv`. **Resumable:** a finished trial (`trial.json` committed in the repo) is skipped, so stages can span Kaggle sessions.

## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order.
- Then clone (or pull) the public repo so the committed `gnn_core.py` is the one imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- `STAGES`: the pre-registered knob order and grids; the winner of each stage is carried into the next.
- `TUNE`: the only change to the v1 recipe before tuning starts — cosine length 100 (early stopping decides the run length).
- `OUT_ROOT` sits **inside the repo clone**, so trials committed after an earlier session are found and skipped.

In [ ]:
OPERATOR = 'gin'
CFG      = dict(mp='base', readout='base')     # config 2 — the configuration every recipe is tuned on
SEEDS    = (42, 43)                             # tuning seeds; the final retrain adds 44
TUNE     = dict(epochs=100)                     # v1 recipe + cosine T_max 100; nothing else changes before stage 1

# one knob at a time, architecture first, then optimizer, regularisation last (EXPERIMENTS.md section 2b);
# the incumbent value is in every grid, so the v1 recipe is the starting incumbent
STAGES = [
    ('s1_norm',       'norm',       ['none', 'batch', 'layer']),
    ('s2_layers',     'num_layers', [2, 4, 6]),
    ('s3_ffn',        'ffn',        [False, True]),
    ('s4_pos_weight', 'pos_weight', [8.0, 16.0]),
    ('s5_lr',         'lr',         [5e-4, 1e-3, 3e-3]),
    ('s6_dropout',    'dropout',    [0.2, 0.3, 0.4]),
]
PATIENCE, MIN_EPOCHS, EVAL_TRAIN_EVERY = 10, 20, 5   # early stopping on the smoothed val F1; train scored every 5th epoch
PROBE_DEPTHS = (2, 4, 6)                             # heaviest setting (BatchNorm + FFN) probed before stage 1

MP_DIRECTION      = 'in'     # fixed for the whole tuning
TEMPORAL_SAMPLING = False

# paths — OUT_ROOT inside the repo clone so committed trials resume
DATA_DIR = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
OUT_ROOT = f'{REPO_DIR}/Outputs/TUNING/{OPERATOR.upper()}'

import gnn_core as core
os.makedirs(OUT_ROOT, exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | gnn_core from {core.__file__}')
print(f'v1 recipe {core.full_recipe(None)}')
print(f'tuning starts from {core.full_recipe(TUNE)}')

## 2. Load graphs

- Three cumulative PyG snapshots; context edges carry label −1, only `eval_mask` edges are scored.
- Only train and validation are used here; the test snapshot is loaded (same loader as every notebook) but never read.

In [ ]:
graphs, col_sets, node_dim = core.load_graphs(DATA_DIR)

## 3. Probe — memory and time per depth

- Per depth (2 / 4 / 6) at the heaviest setting (BatchNorm + FFN): 3 warm-up batches, 20 timed training batches with peak GPU memory, then one full prediction pass over the validation graph.
- A depth passes when it fits and its epoch estimate is ≤ 3× the 2-layer one; otherwise the far-hop fallback list is tried, then the depth is dropped.
- `NEIGHBOURS_USED` is what the stages run with; the full record goes to `probe.json`.

In [ ]:
HEAVIEST = dict(TUNE, norm='batch', ffn=True)
probe = core.probe_depths(OPERATOR, CFG, graphs, col_sets, node_dim, HEAVIEST, PROBE_DEPTHS,
                          f'{OUT_ROOT}/probe.json', mp_direction=MP_DIRECTION, temporal=TEMPORAL_SAMPLING)
NEIGHBOURS_USED = core.neighbours_from_probe(probe)
print('neighbour lists used:', NEIGHBOURS_USED)

## 4. Run the stages

- Per stage: one trial per grid value (both seeds back to back, then the post-trial scoring → `trial.json`), the winner by the pre-registered rule, the edge rule if the winner sits at the grid edge, then the winner becomes the incumbent.
- The incumbent's recipe is never retrained in a later stage: its scoring is copied into that stage.
- An edge-rule depth (3 or 8) is probed first and runs only if it passes.
- Per-epoch line: train / val loss, F1 (train at the val threshold, every 5th epoch), PR-AUC, seconds, RAM; `* new best` marks the checkpoint.

In [ ]:
incumbent = core.full_recipe(TUNE)          # the v1 recipe + T_max 100; each stage may replace one knob
incumbent_trial, winners = None, {}
for stage, knob, grid in STAGES:
    stage_dir = f'{OUT_ROOT}/{stage}'
    os.makedirs(stage_dir, exist_ok=True)
    trials, recipes = [], {}
    for value in grid:
        recipe = core.with_knob(incumbent, knob, value, NEIGHBOURS_USED)
        if recipe is None:
            print(f'{stage}: {knob} = {value} skipped (depth dropped by the probe)')
            continue
        trial = core.run_tuning_trial(OPERATOR, CFG, graphs, col_sets, node_dim, stage_dir, recipe, SEEDS, TUNE,
                                      MP_DIRECTION, TEMPORAL_SAMPLING, EVAL_TRAIN_EVERY, PATIENCE, MIN_EPOCHS,
                                      reuse=incumbent_trial)
        trials.append(trial)
        recipes[trial['trial']] = recipe
    winner = core.choose_winner(trials, core.recipe_tag(incumbent, TUNE))

    # edge rule: a numeric winner at the edge of its grid gets one trial one step beyond
    beyond = core.edge_value(knob, winner['recipe'][knob], grid)
    if beyond is not None:
        if knob == 'num_layers' and beyond not in NEIGHBOURS_USED:
            extra_probe = core.probe_depths(OPERATOR, CFG, graphs, col_sets, node_dim, HEAVIEST, (beyond,),
                                            f'{OUT_ROOT}/probe_{beyond}L.json', mp_direction=MP_DIRECTION,
                                            temporal=TEMPORAL_SAMPLING, reference_seconds=probe['reference_seconds'])
            NEIGHBOURS_USED = core.neighbours_from_probe(extra_probe, NEIGHBOURS_USED)
        recipe = core.with_knob(winner['recipe'], knob, beyond, NEIGHBOURS_USED)
        if recipe is None:
            print(f'{stage}: edge trial {knob} = {beyond} skipped (depth failed the probe)')
        else:
            extra = core.run_tuning_trial(OPERATOR, CFG, graphs, col_sets, node_dim, stage_dir, recipe, SEEDS, TUNE,
                                          MP_DIRECTION, TEMPORAL_SAMPLING, EVAL_TRAIN_EVERY, PATIENCE, MIN_EPOCHS)
            trials.append(extra)
            recipes[extra['trial']] = recipe
            winner = core.choose_winner(trials, winner['trial'])

    incumbent, incumbent_trial, winners[stage] = recipes[winner['trial']], winner, winner['trial']
    core.summarize_stage(stage_dir, winner['trial'])
    print(f">>> after {stage}: incumbent = {winner['trial']} | best epoch {winner['best_epoch']} | {incumbent}")
print('=' * 90)
print('all stages complete | incumbent:', incumbent_trial['trial'], incumbent)

## 5. Summary across the stages

- `trials_summary.csv`: one row per trial with every knob, best / stopped epochs, train and validation F1 / PR-AUC (mean ± std), the train − val gap and the winner flag.
- Gap caveat: validation has 0.107 % laundering vs 0.075 % on train, so part of any gap is prevalence, not underfitting.
- `tuning_curves.png`: seed-mean validation F1 per epoch, one line per trial, winners bold.

In [ ]:
summary = core.summarize_tuning(OUT_ROOT, winners)
summary[['stage', 'trial', 'winner', 'dropout', 'norm', 'num_layers', 'num_neighbors', 'pos_weight', 'lr', 'ffn',
         'best_epoch', 'epochs_run', 'reached_max', 'val_f1_mean', 'val_f1_std', 'train_f1_mean', 'gap_f1',
         'val_pr_auc_mean', 'thresholds', 'params', 'invariant_params']].round(4)

## 6. Freeze the recipe

- Only after every stage has run: `recipe.json` = knobs, neighbour lists, T_max, best epoch, per-seed thresholds for the four retrain seeds (42 and 43 their own; 44 and 45 the mean of those two), the validation F1 that chose them.
- This file is the single source of the frozen GIN recipe; `GIN_tuning_final.ipynb` loads it.

In [ ]:
assert set(winners) == {stage for stage, _, _ in STAGES}, 'run every stage before freezing the recipe'
frozen = core.save_recipe(f'{OUT_ROOT}/recipe.json', OPERATOR, CFG, incumbent_trial)

## 7. Download

- Zip `OUT_ROOT`; extract into `Outputs/TUNING/GIN/` in the repo and commit the small files (`trial.json`, `history.csv`, `curves.png`, summaries, `probe.json`, `recipe.json`; `best.pt` stays out of git).

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_TUNING_GIN', 'zip', OUT_ROOT)